# Algorithmic Trading Backtester — Exploration

This notebook runs all strategies end to end, compares their performance, and visualises the results.

## 1. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt

from data import DataLoader
from backtest import BackTester, BackTestResult
from strategies import (
    MovingAverageCrossover,
    MomentumStrategy,
    MeanReversionStrategy,
    RSIStrategy,
    BollingerBandStrategy,
    MACDStrategy,
    VolatilityBreakout,
    VWAPStrategy,
)
from metrics import summary

print("All imports loaded successfully")

## 2. Load Data

In [ ]:
loader = DataLoader(cache_dir="../data")
data = loader.fetch("AAPL", start="2018-01-01", end="2024-01-01")

print(f"Loaded {len(data)} rows of data")
print(f"Date range: {data.index[0].date()} to {data.index[-1].date()}")
data.head()

## 3. Run a Single Strategy (Quick Sanity Check)

Test one strategy first before running all 8.

In [ ]:
strategy = MovingAverageCrossover(short_window=50, long_window=200)
backtester = BackTester(starting_capital=10_000)
result = backtester.run_backtest(data, strategy)

print("Equity curve (last 5 days):")
print(result.equity_curve.tail())
print(f"\nFinal portfolio value: ${result.equity_curve.iloc[-1]:,.2f}")
print(f"Number of trades: {len(result.trades)}")

## 4. Run All Strategies & Compare

In [ ]:
strategies = {
    "MA Crossover": MovingAverageCrossover(),
    "Momentum": MomentumStrategy(),
    "Mean Reversion": MeanReversionStrategy(),
    "RSI": RSIStrategy(),
    "Bollinger Band": BollingerBandStrategy(),
    "MACD": MACDStrategy(),
    "Volatility Breakout": VolatilityBreakout(),
    "VWAP": VWAPStrategy(),
}

backtester = BackTester(starting_capital=10_000)
results = {}
summaries = {}

for name, strategy in strategies.items():
    result = backtester.run_backtest(data, strategy)
    results[name] = result
    summaries[name] = summary(result)
    print(f"Finished: {name}")

comparison = pd.DataFrame(summaries).T
comparison.round(4)

## 5. Equity Curves — All Strategies vs Buy & Hold

In [ ]:
buy_hold_returns = data["Close"].pct_change().fillna(0)
buy_hold_curve = 10_000 * (1 + buy_hold_returns).cumprod()

plt.figure(figsize=(14, 7))
plt.plot(buy_hold_curve.index, buy_hold_curve, label="Buy & Hold", linewidth=2, color="black", linestyle="--")

for name, result in results.items():
    plt.plot(result.equity_curve.index, result.equity_curve, label=name, alpha=0.7)

plt.title("Strategy Equity Curves vs Buy & Hold (AAPL 2018-2024)")
plt.xlabel("Date")
plt.ylabel("Portfolio Value ($)")
plt.legend(loc="upper left", fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Drawdown — Worst Strategy Periods

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Best performing strategy by Sharpe
best_name = comparison["Sharpe Ratio"].idxmax()
best_result = results[best_name]

# Worst performing strategy by Sharpe
worst_name = comparison["Sharpe Ratio"].idxmin()
worst_result = results[worst_name]

for ax, name, result in [(axes[0], best_name, best_result), (axes[1], worst_name, worst_result)]:
    cummax = result.equity_curve.cummax()
    drawdown = (result.equity_curve - cummax) / cummax
    ax.fill_between(drawdown.index, drawdown, 0, color="red", alpha=0.3)
    ax.plot(drawdown.index, drawdown, color="red", linewidth=1)
    ax.set_title(f"Drawdown: {name}")
    ax.set_ylabel("Drawdown")
    ax.grid(alpha=0.3)

plt.xlabel("Date")
plt.tight_layout()
plt.show()

## 7. Strategy Comparison Bar Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_to_plot = ["Sharpe Ratio", "Total Return", "Max Drawdown"]
colours = ["steelblue", "green", "red"]

for ax, metric, colour in zip(axes, metrics_to_plot, colours):
    comparison[metric].sort_values().plot(kind="barh", ax=ax, color=colour)
    ax.set_title(metric)
    ax.grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

## 8. Test on a Different Ticker

Run the best strategy on a different stock to see if the results hold up.

In [ ]:
msft_data = loader.fetch("MSFT", start="2018-01-01", end="2024-01-01")

best_strategy = strategies[best_name]
msft_result = backtester.run_backtest(msft_data, best_strategy)

msft_buy_hold = 10_000 * (1 + msft_data["Close"].pct_change().fillna(0)).cumprod()

plt.figure(figsize=(12, 6))
plt.plot(msft_result.equity_curve.index, msft_result.equity_curve, label=f"{best_name} Strategy")
plt.plot(msft_buy_hold.index, msft_buy_hold, label="Buy & Hold", linestyle="--", color="black")
plt.title(f"Best Strategy ({best_name}) on MSFT — Does It Generalise?")
plt.xlabel("Date")
plt.ylabel("Portfolio Value ($)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n--- {best_name} on MSFT ---")
print(summary(msft_result).round(4))

## 9. Save Results

In [ ]:
comparison.round(4).to_csv("../comparison_results.csv")
print("Saved comparison_results.csv")
print("\nFinal comparison table:")
comparison.round(4)